# Structured Output — ôn BT1 câu 1

**Mục tiêu:** hiểu ba kiểu dữ liệu trong route extract. Học theo thứ tự từ trên xuống. Các cell chính chỉ cần Python và Pydantic, dùng dữ liệu mẫu. Cell OpenAI ở cuối mặc định tắt.

Ví dụ là một yêu cầu bảo trì. Tên schema khác BT1 nhưng vai trò giống nhau.

## 1. Request — người dùng gửi gì?

Ở BT1, `TicketExtractIn` nhận mã, giờ nhận và nội dung. Dưới đây, `BaseModel` khai báo kiểu dữ liệu. `Field(min_length=1)` yêu cầu chuỗi có ít nhất một ký tự. `AwareDatetime` yêu cầu thời điểm có múi giờ, như `+07:00`.

In [ ]:
from typing import Literal

from pydantic import AwareDatetime, BaseModel, Field


class RequestIn(BaseModel):
    id: str = Field(min_length=1)
    received_at: AwareDatetime
    message: str = Field(min_length=1)


request = RequestIn(
    id="REQ-7",
    received_at="2026-11-09T09:00:00+07:00",
    message="Quạt khu B phát tiếng kêu. Chưa rõ mã thiết bị.",
)
print(request.message)
print(request.received_at)

## 2. Bản nháp — model cần trả gì?

Ở BT1, `TicketExtracted` mô tả phần trích xuất. `Literal` giới hạn các giá trị được chọn. `str | None` cho phép tên thiết bị hoặc `None` khi chưa biết. `description` nói cho model biết ý nghĩa của trường.

Cell này tự tạo một bản nháp mẫu bằng `RequestDraft(...)`. Khi nối model ở bước 4, model sẽ tạo dữ liệu và ta nhận về một đối tượng cùng kiểu.

In [ ]:
class RequestDraft(BaseModel):
    kind: Literal["electrical", "mechanical", "unknown"]
    asset: str | None = Field(description="Mã thiết bị trong tin nhắn, chưa rõ thì trả null.")
    summary: str = Field(max_length=120)


draft = RequestDraft(
    kind="mechanical",
    asset=None,
    summary="Quạt khu B phát tiếng kêu.",
)
print(draft.kind)
print(draft.asset)
print(draft.summary)

## 3. Response — backend ghép gì?

Ở BT1, `TicketExtractOut` có phần nháp và thêm `id`, `received_at`. Cú pháp `class RequestOut(RequestDraft)` cho lớp mới dùng lại các trường của bản nháp. Khi tạo response, lấy mã và giờ từ request ban đầu. Lấy phân loại, thiết bị và tóm tắt từ bản nháp.

In [ ]:
class RequestOut(RequestDraft):
    id: str
    received_at: AwareDatetime


response = RequestOut(
    id=request.id,
    received_at=request.received_at,
    kind=draft.kind,
    asset=draft.asset,
    summary=draft.summary,
)
print(response.id, response.kind, response.summary)

assert response.id == "REQ-7"
assert response.received_at == request.received_at
assert response.asset is None

## 4. Khi nối model, ba dòng sau làm gì?

1. `ChatOpenAI(...)` tạo đối tượng để gọi OpenAI.
2. `llm.with_structured_output(RequestDraft)` đặt cấu trúc cần trả theo schema.
3. `extractor.invoke(...)` gửi hướng dẫn và nội dung lên model. Nếu thành công, bạn nhận một `RequestDraft` và đọc được `result.summary`.

Cell dưới mặc định tắt. Để tự thử, cần `langchain-openai`, `OPENAI_API_KEY` đã đặt trong môi trường và quyền dùng model được chọn. Tên model trong ví dụ lấy từ notebook buổi 4.

In [ ]:
RUN_LLM_DEMO = False

if RUN_LLM_DEMO:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    extractor = llm.with_structured_output(RequestDraft)
    result = extractor.invoke([
        ("system", "Trích xuất yêu cầu bảo trì. Chỉ dùng thông tin trong tin nhắn."),
        ("human", request.message),
    ])
    print(result.summary)

**Tự thử:** đổi `kind="mechanical"` thành `kind="other"` ở bước 2 rồi chạy cell đó. Đọc lỗi Pydantic và khôi phục giá trị. Sau đó trả lời: nếu model đưa ra mã thiết bị trông hợp lệ nhưng vắng trong tin nhắn, ai cần kiểm bản nháp trước khi xác nhận?

Mở tiếp [tool_calling.ipynb](tool_calling.ipynb).